# Batch Evaluation Table

Objetivo deste notebook: preparar e, quando voce quiser, executar a avaliacao final em lote.

Fluxo manual recomendado:
1. Fase A - tabela: deixe `RUN_BATCH_INFERENCE = False` e rode setup + as quatro celulas de dataset. Isso exporta apenas `evaluation/<dataset>/dataset_csv`.
2. Fase B - inferencia: mude `RUN_BATCH_INFERENCE = True` e rode um dataset por vez. Isso carrega modelos, usa `TAU_INTENT`/`TAU_SLOTS` e exporta `evaluation/<dataset>/results_csv`.
3. Fase C - metricas: depois de existir `<dataset>_predictions.csv`, abra `batch_eval_metrics.ipynb`.

Cuidados:
- Este notebook nao deve ser executado inteiro no modo inferencia sem querer; a inferencia e pesada.
- `USE_GOLD_INTENT_FOR_SLOTS=True` faz slot filling com o intent ouro da sentenca.
- Os candidatos de intent sao separados por dataset; slots sao os labels do intent de avaliacao.


In [2]:
# Celula 1 - bootstrap/imports.
# Execute uma vez no inicio. Nao carrega modelos pesados; apenas localiza o repo e importa helpers.

import sys
import json
import time
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src').exists() and (candidate / 'datasets').exists():
            return candidate
    raise RuntimeError('Nao encontrei a raiz do projeto a partir do diretorio atual.')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import display

from src.core.pipeline import initialize_nlu, intent_slot_pipeline
from src.data_tools.batch_eval_utils import (
    build_dataset_and_semantic_views,
    build_semantic_intent_candidates_by_dataset,
    build_semantic_intent_slot_dict,
    compute_semantic_exact_match,
    export_evaluation_csvs,
    load_semantic_alias_registry,
    merge_predictions_into_dataset_view,
    run_semantic_batch_inference,
    validate_registry_against_metadata,
)

print('REPO_ROOT =', REPO_ROOT)

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


REPO_ROOT = E:\Intent_Slot_Code


In [3]:
# Celula 2 - configuracao manual da rodada.
# Fase A: deixe RUN_BATCH_INFERENCE=False para adaptar datasets para dataset_csv sem carregar modelos.
# Fase B: mude RUN_BATCH_INFERENCE=True para rodar inferencia e gerar results_csv, um dataset por vez.

REGISTRY_PATH = REPO_ROOT / 'datasets' / 'metadata' / 'semantic_alias_registry.json'
OUTPUT_DIR = REPO_ROOT / 'evaluation'
SLOT_DELIMITER = ' || '
PREVIEW_ROWS = 5

# False = exporta apenas dataset_csv. True = tambem carrega modelos e exporta results_csv.
RUN_BATCH_INFERENCE = True
# True = slot filling usa o intent ouro da sentenca; recomendado para avaliar slot isolado do erro de intent.
USE_GOLD_INTENT_FOR_SLOTS = True
DEVICE = 0
# Threshold final escolhido para intent recognition.
TAU_INTENT = 0.25
# Threshold final escolhido para associacao de slots.
TAU_SLOTS = 0.35
INTENT_BATCH_SIZE = 16
SPACY_BATCH_SIZE = 64
SPACY_N_PROCESS = 1
QA_BATCH_SIZE = 16
SLOT_SENTENCE_BATCH_SIZE = 8
NLI_BATCH_SIZE = 16

# Benchmark legado opcional. Deixe False na avaliacao final; use manualmente se quiser comparar velocidade.
RUN_LEGACY_BASELINE = False
BENCHMARK_SIZES = [100, 500, 1000]

# SNIPS final esperado: 7 arquivos/intents, 13.784 exemplos.
SNIPS_DATASET_PATHS = [
    REPO_ROOT / 'datasets' / 'snips' / 'raw' / 'train_AddToPlaylist_full.json',
    REPO_ROOT / 'datasets' / 'snips' / 'raw' / 'train_BookRestaurant_full.json',
    REPO_ROOT / 'datasets' / 'snips' / 'raw' / 'train_GetWeather_full.json',
    REPO_ROOT / 'datasets' / 'snips' / 'processed' / 'clean' / 'train_PlayMusic_full_clean.json',
    REPO_ROOT / 'datasets' / 'snips' / 'processed' / 'clean' / 'train_RateBook_full_clean.json',
    REPO_ROOT / 'datasets' / 'snips' / 'processed' / 'clean' / 'train_SearchCreativeWork_full_clean.json',
    REPO_ROOT / 'datasets' / 'snips' / 'processed' / 'clean' / 'train_SearchScreeningEvent_full_clean.json',
]

# MultiWOZ final esperado: 6 arquivos/intents, 12.797 exemplos.
MULTIWOZ_DATASET_PATHS = sorted(
    (REPO_ROOT / 'datasets' / 'multiwoz' / 'processed' / 'with_slots').glob('*.json')
)

# ATIS Min50 final esperado: 7 arquivos/intents, 5.232 exemplos; quantity_min50.json fica fora porque esta vazio.
ATIS_MIN50_DATASET_PATHS = [
    path
    for path in sorted((REPO_ROOT / 'datasets' / 'atis' / 'processed' / 'filtered' / 'min50').glob('*.json'))
    if path.name != 'quantity_min50.json'
]

# Field Operations final esperado: 9 arquivos, 9 intents semanticos, 2.700 exemplos.
FIELD_OPERATIONS_DATASET_PATHS = sorted(
    (REPO_ROOT / 'datasets' / 'field_operations' / 'raw').glob('*.json')
)

DATASET_PATHS_BY_LABEL = {
    'snips': SNIPS_DATASET_PATHS,
    'multiwoz': MULTIWOZ_DATASET_PATHS,
    'atis': ATIS_MIN50_DATASET_PATHS,
    'field_operations': FIELD_OPERATIONS_DATASET_PATHS,
}

for dataset_label, paths in DATASET_PATHS_BY_LABEL.items():
    print(f'{dataset_label}: {len(paths)} arquivo(s)')


snips: 7 arquivo(s)
multiwoz: 6 arquivo(s)
atis: 7 arquivo(s)
field_operations: 9 arquivo(s)


In [4]:
# Celula 3 - funcoes compartilhadas da rodada.
# Sempre execute antes das celulas por dataset. Com RUN_BATCH_INFERENCE=False, nao carrega modelos.
# A funcao run_batch_eval_dataset sempre cria dataset_view/semantic_view e exporta dataset_csv.
# Se RUN_BATCH_INFERENCE=True, ela tambem roda intent+slot batch e exporta results_csv.

def get_nlu_processor():
    global nlu_processor
    if 'nlu_processor' not in globals() or nlu_processor is None:
        nlu_processor = initialize_nlu(device=DEVICE, tau_intent=TAU_INTENT, tau_slots=TAU_SLOTS)
    return nlu_processor


def run_batch_eval_dataset(dataset_label: str, dataset_paths):
    dataset_paths = [Path(path) for path in dataset_paths]
    if not dataset_paths:
        raise ValueError(f'Nenhum arquivo configurado para {dataset_label}.')

    registry = load_semantic_alias_registry(REGISTRY_PATH)
    validate_registry_against_metadata(registry)

    dataset_view, semantic_view = build_dataset_and_semantic_views(
        dataset_paths,
        registry=registry,
        slot_delimiter=SLOT_DELIMITER,
    )

    if dataset_view.empty:
        raise ValueError(f'Dataset view vazio para {dataset_label}.')

    dataset_names = sorted(dataset_view['dataset_name'].drop_duplicates().tolist())
    semantic_intent_slot_dict = build_semantic_intent_slot_dict(dataset_names, registry=registry)
    intent_candidates_by_dataset = build_semantic_intent_candidates_by_dataset(dataset_names, registry=registry)
    written_paths = export_evaluation_csvs(dataset_view=dataset_view, evaluation_root=OUTPUT_DIR)

    print('=' * 80)
    print(f'Batch eval dataset: {dataset_label}')
    print('Arquivos selecionados:', len(dataset_paths))
    print('Datasets detectados:', dataset_names)
    print('Dataset view shape:', dataset_view.shape)
    print('Semantic view shape:', semantic_view.shape)
    print('CSV(s) dataset_csv exportados:')
    for path in written_paths['dataset_csv']:
        print('-', path)

    display(dataset_view.head(PREVIEW_ROWS))
    display(semantic_view.head(PREVIEW_ROWS))

    result = {
        'dataset_label': dataset_label,
        'dataset_paths': dataset_paths,
        'dataset_view': dataset_view,
        'semantic_view': semantic_view,
        'semantic_intent_slot_dict': semantic_intent_slot_dict,
        'intent_candidates_by_dataset': intent_candidates_by_dataset,
        'written_dataset_paths': written_paths['dataset_csv'],
    }

    if not RUN_BATCH_INFERENCE:
        print('RUN_BATCH_INFERENCE=False: inferencia em batch pulada.')
        return result

    processor = get_nlu_processor()
    semantic_predictions, batch_stats = run_semantic_batch_inference(
        processor,
        semantic_view,
        semantic_intent_slot_dict,
        intent_candidates_by_dataset=intent_candidates_by_dataset,
        use_gold_intent_for_slots=USE_GOLD_INTENT_FOR_SLOTS,
        intent_batch_size=INTENT_BATCH_SIZE,
        spacy_batch_size=SPACY_BATCH_SIZE,
        spacy_n_process=SPACY_N_PROCESS,
        qa_batch_size=QA_BATCH_SIZE,
        slot_sentence_batch_size=SLOT_SENTENCE_BATCH_SIZE,
        nli_batch_size=NLI_BATCH_SIZE,
    )
    metrics = compute_semantic_exact_match(semantic_predictions)
    dataset_view_with_preds = merge_predictions_into_dataset_view(dataset_view, semantic_predictions)

    written_prediction_paths = export_evaluation_csvs(
        results_view=dataset_view_with_preds,
        evaluation_root=OUTPUT_DIR,
    )

    print('Batch stats:', json.dumps(batch_stats, indent=2, ensure_ascii=False))
    print('Metrics:', json.dumps(metrics, indent=2, ensure_ascii=False))
    print('CSV(s) results_csv exportados:')
    for path in written_prediction_paths['results_csv']:
        print('-', path)

    display(semantic_predictions.head(PREVIEW_ROWS))
    display(dataset_view_with_preds.head(PREVIEW_ROWS))

    result.update({
        'semantic_predictions': semantic_predictions,
        'batch_stats': batch_stats,
        'metrics': metrics,
        'dataset_view_with_preds': dataset_view_with_preds,
        'written_prediction_paths': written_prediction_paths['results_csv'],
    })
    return result


## SNIPS

Proposito: adaptar/avaliar todos os 7 intents SNIPS. Esperado: 7 arquivos de origem e 13.784 linhas no CSV agregado.

- Fase A (`RUN_BATCH_INFERENCE=False`): gera `evaluation/snips/dataset_csv/snips_dataset.csv` e CSVs por intent.
- Fase B (`RUN_BATCH_INFERENCE=True`): gera `evaluation/snips/results_csv/snips_predictions.csv` e CSVs por intent.


In [5]:
# Celula SNIPS - rode manualmente quando quiser processar SNIPS.
snips_batch = run_batch_eval_dataset('snips', SNIPS_DATASET_PATHS)


Batch eval dataset: snips
Arquivos selecionados: 7
Datasets detectados: ['snips']
Dataset view shape: (13784, 48)
Semantic view shape: (13784, 98)
CSV(s) dataset_csv exportados:
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\snips_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\AddToPlaylist_full_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\BookRestaurant_full_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\GetWeather_full_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\PlayMusic_full_clean_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\RateBook_full_clean_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\SearchCreativeWork_full_clean_dataset.csv
- E:\Intent_Slot_Code\evaluation\snips\dataset_csv\SearchScreeningEvent_full_clean_dataset.csv


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,music_item,playlist,artist,playlist_owner,entity_name,cuisine,party_size_number,restaurant_name,timeRange,restaurant_type,party_size_description,spatial_relation,served_dish,sort,facility,poi,city,state,country,geographic_poi,current_location,condition_temperature,condition_description,service,year,track,album,genre,rating_value,rating_unit,best_rating,object_select,object_name,object_type,object_part_of_series_type,movie_name,movie_type,location_name,object_location_type,gold_slots_dataset_json,gold_slots_semantic_json
0,0,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add another song to the Cita Romántica playlist.,False,song,Cita Romántica,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""music_item"": [""song""], ""playlist"": [""Cita Romántica""]}","{""playable content"": [""song""], ""playlist title"": [""Cita Romántica""]}"
1,1,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add clem burke in my playlist Pre-Party R&B Jams,False,,Pre-Party R&B Jams,clem burke,my,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""artist"": [""clem burke""], ""playlist_owner"": [""my""], ""playlist"": [""Pre-Party R&B Jams""]}","{""artist name"": [""clem burke""], ""playlist owner"": [""my""], ""playlist title"": [""Pre-Party R&B Jams""]}"
2,2,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add Live from Aragon Ballroom to Trapeo,False,,Trapeo,,,Live from Aragon Ballroom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""entity_name"": [""Live from Aragon Ballroom""], ""playlist"": [""Trapeo""]}","{""entity name"": [""Live from Aragon Ballroom""], ""playlist title"": [""Trapeo""]}"
3,3,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add Unite and Win to my night out,False,,night out,,my,Unite and Win,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""entity_name"": [""Unite and Win""], ""playlist_owner"": [""my""], ""playlist"": [""night out""]}","{""entity name"": [""Unite and Win""], ""playlist owner"": [""my""], ""playlist title"": [""night out""]}"
4,4,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add track to my Digster Future Hits,False,track,Digster Future Hits,,my,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""music_item"": [""track""], ""playlist_owner"": [""my""], ""playlist"": [""Digster Future Hits""]}","{""playable content"": [""track""], ""playlist owner"": [""my""], ""playlist title"": [""Digster Future Hits""]}"


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,playable content,playlist title,artist name,playlist owner,entity name,cuisine type,number of people,restaurant name,reservation time,restaurant type,party size description,spatial relation,served dish,sorting preference,facility,point of interest,city,state,country,date,geographic point,current location,temperature condition,weather description,music service,year,track title,album title,music genre,rating value,rating unit,best rating,object select,object name,object type,object part of series type,movie title,screening object type,movie type,location name,location type,time range,pred__playable content,pred__playlist title,pred__artist name,pred__playlist owner,pred__entity name,pred__cuisine type,pred__number of people,pred__restaurant name,pred__reservation time,pred__restaurant type,pred__party size description,pred__spatial relation,pred__served dish,pred__sorting preference,pred__facility,pred__point of interest,pred__city,pred__state,pred__country,pred__date,pred__geographic point,pred__current location,pred__temperature condition,pred__weather description,pred__music service,pred__year,pred__track title,pred__album title,pred__music genre,pred__rating value,pred__rating unit,pred__best rating,pred__object select,pred__object name,pred__object type,pred__object part of series type,pred__movie title,pred__screening object type,pred__movie type,pred__location name,pred__location type,pred__time range
0,0,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add another song to the Cita Romántica playlist.,False,"{""music_item"": [""song""], ""playlist"": [""Cita Romántica""]}","{""playable content"": [""song""], ""playlist title"": [""Cita Romántica""]}",,0.0,Add to a playlist,{},{},song,Cita Romántica,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add clem burke in my playlist Pre-Party R&B Jams,False,"{""artist"": [""clem burke""], ""playlist_owner"": [""my""], ""playlist"": [""Pre-Party R&B Jams""]}","{""artist name"": [""clem burke""], ""playlist owner"": [""my""], ""playlist title"": [""Pre-Party R&B Jams""]}",,0.0,Add to a playlist,{},{},,Pre-Party R&B Jams,clem burke,my,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add Live from Aragon Ballroom to Trapeo,False,"{""entity_name"": [""Live from Aragon Ballroom""], ""playlist"": [""Trapeo""]}","{""entity name"": [""Live from Aragon Ballroom""], ""playlist title"": [""Trapeo""]}",,0.0,Add to a playlist,{},{},,Trapeo,,,Live from Aragon Ballroom,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add Unite and Win to my night out,False,"{""entity_name"": [""Unite and Win""], ""playlist_owner"": [""my""], ""playlist"": [""night out""]}","{""entity name"": [""Unite and Win""], ""playlist owner"": [""my""], ""playlist title"": [""night out""]}",,0.0,Add to a playlist,{},{},,night out,,my,Unite and Win,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,4,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add track to my Digster Future Hits,False,"{""music_item"": [""track""], ""playlist_owner"": [""my""], ""playlist"": [""Digster Future Hits""]}","{""playable content"": [""track""], ""playlist owner"": [""my""], ""playlist title"": [""Digster Future Hits""]}",,0.0,Add to a playlist,{},{},track,Digster Future Hits,,my,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,

Loading zero-shot classifier: MoritzLaurer/ModernBERT-large-zeroshot-v2.0...


Device set to use cuda:0


...

10:59:26.370 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading lookup file directly from disk (SLOW) - try index.meta.index-source=fileinmem in the index properties file. 477,6 MiB of memory would be required.
10:59:26.396 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 3,1 GiB of memory would be required.
Índice carregado:
Number of documents: 62603369
Number of terms: 9903087
Number of postings: 191798780
Number of fields: 0
Number of tokens: 192671150
Field names: []
Positions:   false

10:59:31.240 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading lookup file directly from disk (SLOW) - try index.meta.index-source=fileinmem in the index properties file. 477,6 MiB of memory would be required.
10:59:31.242 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Stru

e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  

Batch stats: {
  "intent_seconds": 307.2362003999997,
  "slot_seconds": 11305.627331799999,
  "total_seconds": 11612.863532199997,
  "n_rows": 13784,
  "rows_per_second": 1.186959612655389,
  "group_sizes": {
    "Add to a playlist": 1942,
    "Book a restaurant": 1973,
    "Check weather": 2000,
    "Play a music": 2000,
    "Rate a item": 1956,
    "Search a creative work": 1954,
    "Search for movie showtimes": 1959
  },
  "tau_intent": 0.25,
  "tau_slots": 0.35,
  "intent_candidate_counts_by_dataset": {
    "snips": 7
  }
}
Metrics: {
  "rows": 13784,
  "slot_exact_match": 0.17157573998839235,
  "joint_intent_slot_exact_match": 0.1580818340104469,
  "slot_exact_match_count": 2365,
  "joint_exact_match_count": 2179
}
CSV(s) results_csv exportados:
- E:\Intent_Slot_Code\evaluation\snips\results_csv\snips_predictions.csv
- E:\Intent_Slot_Code\evaluation\snips\results_csv\AddToPlaylist_full_predictions.csv
- E:\Intent_Slot_Code\evaluation\snips\results_csv\BookRestaurant_full_predicti

,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,playable content,playlist title,artist name,playlist owner,entity name,cuisine type,number of people,restaurant name,reservation time,restaurant type,party size description,spatial relation,served dish,sorting preference,facility,point of interest,city,state,country,date,geographic point,current location,temperature condition,weather description,music service,year,track title,album title,music genre,rating value,rating unit,best rating,object select,object name,object type,object part of series type,movie title,screening object type,movie type,location name,location type,time range,pred__playable content,pred__playlist title,pred__artist name,pred__playlist owner,pred__entity name,pred__cuisine type,pred__number of people,pred__restaurant name,pred__reservation time,pred__restaurant type,pred__party size description,pred__spatial relation,pred__served dish,pred__sorting preference,pred__facility,pred__point of interest,pred__city,pred__state,pred__country,pred__date,pred__geographic point,pred__current location,pred__temperature condition,pred__weather description,pred__music service,pred__year,pred__track title,pred__album title,pred__music genre,pred__rating value,pred__rating unit,pred__best rating,pred__object select,pred__object name,pred__object type,pred__object part of series type,pred__movie title,pred__screening object type,pred__movie type,pred__location name,pred__location type,pred__time range
0,0,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add another song to the Cita Romántica playlist.,False,"{""music_item"": [""song""], ""playlist"": [""Cita Romántica""]}","{""playable content"": [""song""], ""playlist title"": [""Cita Romántica""]}",Add to a playlist,0.803,Add to a playlist,"{""playlist title"": [""Cita Romántica""], ""entity name"": [""song""]}","{""playlist title"": 1.468, ""entity name"": 0.997, ""playable content"": 0.0, ""playlist owner"": 0.0, ""artist name"": 0.0}",song,Cita Romántica,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Cita Romántica,,,song,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add clem burke in my playlist Pre-Party R&B Jams,False,"{""artist"": [""clem burke""], ""playlist_owner"": [""my""], ""playlist"": [""Pre-Party R&B Jams""]}","{""artist name"": [""clem burke""], ""playlist owner"": [""my""], ""playlist title"": [""Pre-Party R&B Jams""]}",Add to a playlist,0.772,Add to a playlist,"{""playable content"": [""my""], ""playlist title"": [""Pre-Party R&B Jams""], ""entity name"": [""clem burke""]}","{""playable content"": 0.951, ""playlist title"": 1.8, ""entity name"": 1.832, ""playlist owner"": 0.0, ""artist name"": 0.0}",,Pre-Party R&B Jams,clem burke,my,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,my,Pre-Party R&B Jams,,,clem burke,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add Live from Aragon Ballroom to Trapeo,False,"{""entity_name"": [""Live from Aragon Ballroom""], ""playlist"": [""Trapeo""]}","{""entity name"": [""Live from Aragon Ballroom""], ""playlist title"": [""Trapeo""]}",Play a music,0.386,Add to a playlist,"{""playable content"": [""Live from Aragon Ballroom""], ""entity name"": [""Trapeo""]}","{""playable content"": 1.073, ""entity name"": 0.99, ""playlist title"": 0.0, ""playlist owner"": 0.0, ""artist name"": 0.0}",,Trapeo,,,Live from Aragon Ballroom,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Live from Aragon Ballroom,,,,Trapeo,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add Unite and Win to my night out,False,"{""entity_name"": [""Unite and Win""], ""

,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,music_item,playlist,artist,playlist_owner,entity_name,cuisine,party_size_number,restaurant_name,timeRange,restaurant_type,party_size_description,spatial_relation,served_dish,sort,facility,poi,city,state,country,geographic_poi,current_location,condition_temperature,condition_description,service,year,track,album,genre,rating_value,rating_unit,best_rating,object_select,object_name,object_type,object_part_of_series_type,movie_name,movie_type,location_name,object_location_type,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,pred__playable content,pred__playlist title,pred__artist name,pred__playlist owner,pred__entity name,pred__cuisine type,pred__number of people,pred__restaurant name,pred__reservation time,pred__restaurant type,pred__party size description,pred__spatial relation,pred__served dish,pred__sorting preference,pred__facility,pred__point of interest,pred__city,pred__state,pred__country,pred__date,pred__geographic point,pred__current location,pred__temperature condition,pred__weather description,pred__music service,pred__year,pred__track title,pred__album title,pred__music genre,pred__rating value,pred__rating unit,pred__best rating,pred__object select,pred__object name,pred__object type,pred__object part of series type,pred__movie title,pred__screening object type,pred__movie type,pred__location name,pred__location type,pred__time range
0,0,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add another song to the Cita Romántica playlist.,False,song,Cita Romántica,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""music_item"": [""song""], ""playlist"": [""Cita Romántica""]}","{""playable content"": [""song""], ""playlist title"": [""Cita Romántica""]}",Add to a playlist,0.803,Add to a playlist,"{""playlist title"": [""Cita Romántica""], ""entity name"": [""song""]}","{""playlist title"": 1.468, ""entity name"": 0.997, ""playable content"": 0.0, ""playlist owner"": 0.0, ""artist name"": 0.0}",,Cita Romántica,,,song,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,add clem burke in my playlist Pre-Party R&B Jams,False,,Pre-Party R&B Jams,clem burke,my,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""artist"": [""clem burke""], ""playlist_owner"": [""my""], ""playlist"": [""Pre-Party R&B Jams""]}","{""artist name"": [""clem burke""], ""playlist owner"": [""my""], ""playlist title"": [""Pre-Party R&B Jams""]}",Add to a playlist,0.772,Add to a playlist,"{""playable content"": [""my""], ""playlist title"": [""Pre-Party R&B Jams""], ""entity name"": [""clem burke""]}","{""playable content"": 0.951, ""playlist title"": 1.8, ""entity name"": 1.832, ""playlist owner"": 0.0, ""artist name"": 0.0}",my,Pre-Party R&B Jams,,,clem burke,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,snips,datasets/snips/raw/train_AddToPlaylist_full.json,AddToPlaylist,Add to a playlist,Add Live from Aragon Ballroom to Trapeo,False,,Trapeo,,,Live from Aragon Ballroom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""entity_name"": [""Live from Aragon Ballroom""], ""playlist"": [""Trapeo""]}","{""entity name"": [""Live from Aragon Ballroom""], ""playlist title"": [""Trapeo""]}",Play a music,0.386,Add to a playlist,"{""playable content"": [""Live from Aragon Ballroom""], ""entity name"": [""Trapeo""]}","{""playable content"": 1.073, ""entity name"": 0.99, ""playlist title"": 0.0, ""playlist owner"": 0.0, ""artist name"": 0.0}",Live from Aragon Ballroom,,,,Trapeo,,,,,,,,,,,,,,,,,,,,,,,,,,

## MultiWOZ

Proposito: adaptar/avaliar todos os 6 intents MultiWOZ com slots. Esperado: 6 arquivos de origem e 12.797 linhas no CSV agregado.

- Fase A (`RUN_BATCH_INFERENCE=False`): gera `evaluation/multiwoz/dataset_csv/multiwoz_dataset.csv` e CSVs por intent.
- Fase B (`RUN_BATCH_INFERENCE=True`): gera `evaluation/multiwoz/results_csv/multiwoz_predictions.csv` e CSVs por intent.


In [6]:
# Celula MultiWOZ - rode manualmente quando quiser processar MultiWOZ.
multiwoz_batch = run_batch_eval_dataset('multiwoz', MULTIWOZ_DATASET_PATHS)


Batch eval dataset: multiwoz
Arquivos selecionados: 6
Datasets detectados: ['multiwoz']
Dataset view shape: (12797, 31)
Semantic view shape: (12797, 44)
CSV(s) dataset_csv exportados:
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\multiwoz_dataset.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\slots_attraction__find_attraction_dataset.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\slots_hospital__find_hospital_dataset.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\slots_hotel__find_hotel_dataset.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\slots_restaurant__find_restaurant_dataset.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\slots_taxi__find_taxi_dataset.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\dataset_csv\slots_train__find_train_dataset.csv


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,attraction_type,attraction_area,attraction_name,hospital_department,hotel_type,hotel_name,hotel_area,hotel_pricerange,hotel_stars,restaurant_food,restaurant_area,restaurant_pricerange,restaurant_name,taxi_destination,taxi_departure,taxi_leaveat,taxi_arriveby,train_destination,train_departure,train_arriveby,train_leaveat,train_day,gold_slots_dataset_json,gold_slots_semantic_json
0,0,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,I am getting ready to go off to college. I thought while I was in town maybe I could visit one.,False,college,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}"
1,1,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,What fun places can I visit in the East?,False,,East,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_area"": [""East""]}","{""area"": [""East""]}"
2,2,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am looking for a place to go in the centre, of type multiple sports.",False,multiple sports,centre,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_area"": [""centre""], ""attraction_type"": [""multiple sports""]}","{""area"": [""centre""], ""attraction type"": [""multiple sports""]}"
3,3,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am traveling to Cambridge and excited about seeing location attractions. Could you help me find a place to go, like a college?",False,college,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}"
4,4,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,Can you help me find a place to go on the east side of town where I can enjoy multiple sports? Thanks.,False,multiple sports,east,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_area"": [""east""], ""attraction_type"": [""multiple sports""]}","{""area"": [""east""], ""attraction type"": [""multiple sports""]}"


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,attraction type,area,attraction name,hospital department,hotel type,hotel name,price range,star rating,cuisine type,restaurant name,destination,departure location,leave at time,arrive by time,travel day,pred__attraction type,pred__area,pred__attraction name,pred__hospital department,pred__hotel type,pred__hotel name,pred__price range,pred__star rating,pred__cuisine type,pred__restaurant name,pred__destination,pred__departure location,pred__leave at time,pred__arrive by time,pred__travel day
0,0,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,I am getting ready to go off to college. I thought while I was in town maybe I could visit one.,False,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}",,0.0,Find a tourist attraction,{},{},college,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,What fun places can I visit in the East?,False,"{""attraction_area"": [""East""]}","{""area"": [""East""]}",,0.0,Find a tourist attraction,{},{},,East,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am looking for a place to go in the centre, of type multiple sports.",False,"{""attraction_area"": [""centre""], ""attraction_type"": [""multiple sports""]}","{""area"": [""centre""], ""attraction type"": [""multiple sports""]}",,0.0,Find a tourist attraction,{},{},multiple sports,centre,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am traveling to Cambridge and excited about seeing location attractions. Could you help me find a place to go, like a college?",False,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}",,0.0,Find a tourist attraction,{},{},college,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,4,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,Can you help me find a place to go on the east side of town where I can enjoy multiple sports? Thanks.,False,"{""attraction_area"": [""east""], ""attraction_type"": [""multiple sports""]}","{""area"": [""east""], ""attraction type"": [""multiple sports""]}",,0.0,Find a tourist attraction,{},{},multiple sports,east,,,,,,,,,,,,,,,,,,,,,,,,,,,,


e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: 

Batch stats: {
  "intent_seconds": 271.8719667999976,
  "slot_seconds": 6417.552017800001,
  "total_seconds": 6689.423984599998,
  "n_rows": 12797,
  "rows_per_second": 1.9130197203018537,
  "group_sizes": {
    "Find a tourist attraction": 1179,
    "Find a hospital": 75,
    "Find a hotel": 1594,
    "Find a restaurant": 2415,
    "Find a taxi": 380,
    "Find a train": 7154
  },
  "tau_intent": 0.25,
  "tau_slots": 0.35,
  "intent_candidate_counts_by_dataset": {
    "multiwoz": 6
  }
}
Metrics: {
  "rows": 12797,
  "slot_exact_match": 0.19309213096819566,
  "joint_intent_slot_exact_match": 0.14714386184261935,
  "slot_exact_match_count": 2471,
  "joint_exact_match_count": 1883
}
CSV(s) results_csv exportados:
- E:\Intent_Slot_Code\evaluation\multiwoz\results_csv\multiwoz_predictions.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\results_csv\slots_attraction__find_attraction_predictions.csv
- E:\Intent_Slot_Code\evaluation\multiwoz\results_csv\slots_hospital__find_hospital_predictions

,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,attraction type,area,attraction name,hospital department,hotel type,hotel name,price range,star rating,cuisine type,restaurant name,destination,departure location,leave at time,arrive by time,travel day,pred__attraction type,pred__area,pred__attraction name,pred__hospital department,pred__hotel type,pred__hotel name,pred__price range,pred__star rating,pred__cuisine type,pred__restaurant name,pred__destination,pred__departure location,pred__leave at time,pred__arrive by time,pred__travel day
0,0,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,I am getting ready to go off to college. I thought while I was in town maybe I could visit one.,False,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}",Find a tourist attraction,0.819,Find a tourist attraction,"{""attraction type"": [""college""], ""area"": [""town""]}","{""attraction type"": 1.673, ""area"": 0.974, ""attraction name"": 0.0}",college,,,,,,,,,,,,,,,college,town,,,,,,,,,,,,,
1,1,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,What fun places can I visit in the East?,False,"{""attraction_area"": [""East""]}","{""area"": [""East""]}",Find a tourist attraction,0.981,Find a tourist attraction,"{""attraction type"": [""fun""], ""area"": [""East""]}","{""attraction type"": 0.951, ""area"": 1.001, ""attraction name"": 0.0}",,East,,,,,,,,,,,,,,fun,East,,,,,,,,,,,,,
2,2,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am looking for a place to go in the centre, of type multiple sports.",False,"{""attraction_area"": [""centre""], ""attraction_type"": [""multiple sports""]}","{""area"": [""centre""], ""attraction type"": [""multiple sports""]}",Find a restaurant,0.382,Find a tourist attraction,"{""attraction type"": [""multiple sports""], ""area"": [""looking for a place""], ""attraction name"": [""centre""]}","{""attraction type"": 1.824, ""area"": 0.971, ""attraction name"": 0.972}",multiple sports,centre,,,,,,,,,,,,,,multiple sports,looking for a place,centre,,,,,,,,,,,,
3,3,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am traveling to Cambridge and excited about seeing location attractions. Could you help me find a place to go, like a college?",False,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}",Find a tourist attraction,0.954,Find a tourist attraction,"{""attraction type"": [""college""], ""area"": [""find a place""], ""attraction name"": [""Cambridge""]}","{""attraction type"": 1.385, ""area"": 0.928, ""attraction name"": 0.994}",college,,,,,,,,,,,,,,,college,find a place,Cambridge,,,,,,,,,,,,
4,4,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,Can you help me find a place to go on the east side of town where I can enjoy multiple sports? Thanks.,False,"{""attraction_area"": [""east""], ""attraction_type"": [""multiple sports""]}","{""area"": [""east""], ""attraction type"": [""multiple sports""]}",Find a tourist attraction,0.436,Find a tourist attraction,"{""attraction type"": [""multiple sports""], ""area"": [""place""], ""attraction name"": [""east side""]}","{""attraction type"": 1.65, ""area"": 0.946, ""attraction name"": 0.911}",multiple sports,east,,,,,,,,,,,,,,multiple sports,place,east side,,,,,,,,,,,,


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,attraction_type,attraction_area,attraction_name,hospital_department,hotel_type,hotel_name,hotel_area,hotel_pricerange,hotel_stars,restaurant_food,restaurant_area,restaurant_pricerange,restaurant_name,taxi_destination,taxi_departure,taxi_leaveat,taxi_arriveby,train_destination,train_departure,train_arriveby,train_leaveat,train_day,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,pred__attraction type,pred__area,pred__attraction name,pred__hospital department,pred__hotel type,pred__hotel name,pred__price range,pred__star rating,pred__cuisine type,pred__restaurant name,pred__destination,pred__departure location,pred__leave at time,pred__arrive by time,pred__travel day
0,0,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,I am getting ready to go off to college. I thought while I was in town maybe I could visit one.,False,college,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}",Find a tourist attraction,0.819,Find a tourist attraction,"{""attraction type"": [""college""], ""area"": [""town""]}","{""attraction type"": 1.673, ""area"": 0.974, ""attraction name"": 0.0}",college,town,,,,,,,,,,,,,
1,1,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,What fun places can I visit in the East?,False,,East,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_area"": [""East""]}","{""area"": [""East""]}",Find a tourist attraction,0.981,Find a tourist attraction,"{""attraction type"": [""fun""], ""area"": [""East""]}","{""attraction type"": 0.951, ""area"": 1.001, ""attraction name"": 0.0}",fun,East,,,,,,,,,,,,,
2,2,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am looking for a place to go in the centre, of type multiple sports.",False,multiple sports,centre,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_area"": [""centre""], ""attraction_type"": [""multiple sports""]}","{""area"": [""centre""], ""attraction type"": [""multiple sports""]}",Find a restaurant,0.382,Find a tourist attraction,"{""attraction type"": [""multiple sports""], ""area"": [""looking for a place""], ""attraction name"": [""centre""]}","{""attraction type"": 1.824, ""area"": 0.971, ""attraction name"": 0.972}",multiple sports,looking for a place,centre,,,,,,,,,,,,
3,3,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,"I am traveling to Cambridge and excited about seeing location attractions. Could you help me find a place to go, like a college?",False,college,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_type"": [""college""]}","{""attraction type"": [""college""]}",Find a tourist attraction,0.954,Find a tourist attraction,"{""attraction type"": [""college""], ""area"": [""find a place""], ""attraction name"": [""Cambridge""]}","{""attraction type"": 1.385, ""area"": 0.928, ""attraction name"": 0.994}",college,find a place,Cambridge,,,,,,,,,,,,
4,4,multiwoz,datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json,attraction__find_attraction,Find a tourist attraction,Can you help me find a place to go on the east side of town where I can enjoy multiple sports? Thanks.,False,multiple sports,east,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""attraction_area"": [""east""], ""attraction_type"": [""multiple sports""]}","{""area"": [""

## ATIS Min50

Proposito: adaptar/avaliar os intents ATIS filtrados por min50. Esperado: 7 arquivos de origem e 5.232 linhas no CSV agregado.

- `quantity_min50.json` e excluido de proposito porque esta vazio.
- Fase A (`RUN_BATCH_INFERENCE=False`): gera `evaluation/atis/dataset_csv/atis_dataset.csv` e CSVs por intent.
- Fase B (`RUN_BATCH_INFERENCE=True`): gera `evaluation/atis/results_csv/atis_predictions.csv` e CSVs por intent.


In [7]:

# Celula ATIS - rode manualmente quando quiser processar ATIS Min50.
atis_batch = run_batch_eval_dataset('atis', ATIS_MIN50_DATASET_PATHS)


Batch eval dataset: atis
Arquivos selecionados: 7
Datasets detectados: ['atis']
Dataset view shape: (5232, 40)
Semantic view shape: (5232, 76)
CSV(s) dataset_csv exportados:
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\atis_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\abbreviation_min50_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\aircraft_min50_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\airfare_min50_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\airline_min50_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\flight_min50_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\flight_time_min50_dataset.csv
- E:\Intent_Slot_Code\evaluation\atis\dataset_csv\ground_service_min50_dataset.csv


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,fare_basis_code,fromloc_city_name,toloc_city_name,round_trip,cost_relative,class_type,airline_name,airline_code,fromloc_state_code,fromloc_state_name,toloc_state_code,toloc_state_name,stoploc_city_name,depart_date_day_name,depart_date_day_number,depart_date_month_name,depart_date_today_relative,depart_date_date_relative,depart_time_period_of_day,depart_time_time,depart_time_time_relative,arrive_date_day_name,arrive_time_period_of_day,arrive_time_time,arrive_time_time_relative,flight_mod,flight_stop,meal_description,or,flight_time,city_name,gold_slots_dataset_json,gold_slots_semantic_json
0,0,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code h,False,h,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""h""]}","{""fare basis code"": [""h""]}"
1,1,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is booking class c,False,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""c""]}","{""fare basis code"": [""c""]}"
2,2,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does fare code q mean,False,q,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""q""]}","{""fare basis code"": [""q""]}"
3,3,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code qw,False,qw,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""qw""]}","{""fare basis code"": [""qw""]}"
4,4,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does the fare code f mean,False,f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""f""]}","{""fare basis code"": [""f""]}"


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,fare basis code,departure city,destination city,round trip,cost constraint,travel class,airline name,airline code,departure state code,departure state name,destination state code,destination state name,stopover city,departure day of week,departure day number,departure month,relative to today,relative departure date,departure time of day,departure time,relative departure time,arrival day of week,arrival time of day,arrival time,relative arrival time,flight constraint,number of stops,meal description,alternative marker,flight duration,city,pred__fare basis code,pred__departure city,pred__destination city,pred__round trip,pred__cost constraint,pred__travel class,pred__airline name,pred__airline code,pred__departure state code,pred__departure state name,pred__destination state code,pred__destination state name,pred__stopover city,pred__departure day of week,pred__departure day number,pred__departure month,pred__relative to today,pred__relative departure date,pred__departure time of day,pred__departure time,pred__relative departure time,pred__arrival day of week,pred__arrival time of day,pred__arrival time,pred__relative arrival time,pred__flight constraint,pred__number of stops,pred__meal description,pred__alternative marker,pred__flight duration,pred__city
0,0,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code h,False,"{""fare_basis_code"": [""h""]}","{""fare basis code"": [""h""]}",,0.0,Get an abbreviation meaning,{},{},h,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is booking class c,False,"{""fare_basis_code"": [""c""]}","{""fare basis code"": [""c""]}",,0.0,Get an abbreviation meaning,{},{},c,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does fare code q mean,False,"{""fare_basis_code"": [""q""]}","{""fare basis code"": [""q""]}",,0.0,Get an abbreviation meaning,{},{},q,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code qw,False,"{""fare_basis_code"": [""qw""]}","{""fare basis code"": [""qw""]}",,0.0,Get an abbreviation meaning,{},{},qw,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,4,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does the fare code f mean,False,"{""fare_basis_code"": [""f""]}","{""fare basis code"": [""f""]}",,0.0,Get an abbreviation meaning,{},{},f,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: 

Batch stats: {
  "intent_seconds": 131.15506050000113,
  "slot_seconds": 14854.766661100002,
  "total_seconds": 14985.921721600003,
  "n_rows": 5232,
  "rows_per_second": 0.34912767443986054,
  "group_sizes": {
    "Get an abbreviation meaning": 77,
    "Find aircraft type for a flight": 64,
    "Get airfare information": 465,
    "Find airlines for a route": 123,
    "Find a flight": 4241,
    "Get flight duration": 54,
    "Find ground transportation": 208
  },
  "tau_intent": 0.25,
  "tau_slots": 0.35,
  "intent_candidate_counts_by_dataset": {
    "atis": 7
  }
}
Metrics: {
  "rows": 5232,
  "slot_exact_match": 0.09862385321100918,
  "joint_intent_slot_exact_match": 0.08409785932721713,
  "slot_exact_match_count": 516,
  "joint_exact_match_count": 440
}
CSV(s) results_csv exportados:
- E:\Intent_Slot_Code\evaluation\atis\results_csv\atis_predictions.csv
- E:\Intent_Slot_Code\evaluation\atis\results_csv\abbreviation_min50_predictions.csv
- E:\Intent_Slot_Code\evaluation\atis\results_

,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,fare basis code,departure city,destination city,round trip,cost constraint,travel class,airline name,airline code,departure state code,departure state name,destination state code,destination state name,stopover city,departure day of week,departure day number,departure month,relative to today,relative departure date,departure time of day,departure time,relative departure time,arrival day of week,arrival time of day,arrival time,relative arrival time,flight constraint,number of stops,meal description,alternative marker,flight duration,city,pred__fare basis code,pred__departure city,pred__destination city,pred__round trip,pred__cost constraint,pred__travel class,pred__airline name,pred__airline code,pred__departure state code,pred__departure state name,pred__destination state code,pred__destination state name,pred__stopover city,pred__departure day of week,pred__departure day number,pred__departure month,pred__relative to today,pred__relative departure date,pred__departure time of day,pred__departure time,pred__relative departure time,pred__arrival day of week,pred__arrival time of day,pred__arrival time,pred__relative arrival time,pred__flight constraint,pred__number of stops,pred__meal description,pred__alternative marker,pred__flight duration,pred__city
0,0,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code h,False,"{""fare_basis_code"": [""h""]}","{""fare basis code"": [""h""]}",Get airfare information,0.501,Get an abbreviation meaning,"{""fare basis code"": [""h""]}","{""fare basis code"": 1.072}",h,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,h,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is booking class c,False,"{""fare_basis_code"": [""c""]}","{""fare basis code"": [""c""]}",Get airfare information,0.400,Get an abbreviation meaning,"{""fare basis code"": [""what""]}","{""fare basis code"": 0.902}",c,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,what,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does fare code q mean,False,"{""fare_basis_code"": [""q""]}","{""fare basis code"": [""q""]}",Get an abbreviation meaning,0.899,Get an abbreviation meaning,"{""fare basis code"": [""q""]}","{""fare basis code"": 1.28}",q,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,q,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code qw,False,"{""fare_basis_code"": [""qw""]}","{""fare basis code"": [""qw""]}",Get an abbreviation meaning,0.584,Get an abbreviation meaning,"{""fare basis code"": [""qw""]}","{""fare basis code"": 0.873}",qw,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,qw,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,4,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does the fare code f mean,False,"{""fare_basis_code"": [""f""]}","{""fare basis code"": [""f""]}",Get an abbreviation meaning,0.935,Get an abbreviation meaning,"{""fare basis code"": [""f""]}","{""fare basis code"": 1.014}",f,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,f,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,fare_basis_code,fromloc_city_name,toloc_city_name,round_trip,cost_relative,class_type,airline_name,airline_code,fromloc_state_code,fromloc_state_name,toloc_state_code,toloc_state_name,stoploc_city_name,depart_date_day_name,depart_date_day_number,depart_date_month_name,depart_date_today_relative,depart_date_date_relative,depart_time_period_of_day,depart_time_time,depart_time_time_relative,arrive_date_day_name,arrive_time_period_of_day,arrive_time_time,arrive_time_time_relative,flight_mod,flight_stop,meal_description,or,flight_time,city_name,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,pred__fare basis code,pred__departure city,pred__destination city,pred__round trip,pred__cost constraint,pred__travel class,pred__airline name,pred__airline code,pred__departure state code,pred__departure state name,pred__destination state code,pred__destination state name,pred__stopover city,pred__departure day of week,pred__departure day number,pred__departure month,pred__relative to today,pred__relative departure date,pred__departure time of day,pred__departure time,pred__relative departure time,pred__arrival day of week,pred__arrival time of day,pred__arrival time,pred__relative arrival time,pred__flight constraint,pred__number of stops,pred__meal description,pred__alternative marker,pred__flight duration,pred__city
0,0,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code h,False,h,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""h""]}","{""fare basis code"": [""h""]}",Get airfare information,0.501,Get an abbreviation meaning,"{""fare basis code"": [""h""]}","{""fare basis code"": 1.072}",h,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is booking class c,False,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""c""]}","{""fare basis code"": [""c""]}",Get airfare information,0.400,Get an abbreviation meaning,"{""fare basis code"": [""what""]}","{""fare basis code"": 0.902}",what,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does fare code q mean,False,q,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""q""]}","{""fare basis code"": [""q""]}",Get an abbreviation meaning,0.899,Get an abbreviation meaning,"{""fare basis code"": [""q""]}","{""fare basis code"": 1.28}",q,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what is fare code qw,False,qw,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""qw""]}","{""fare basis code"": [""qw""]}",Get an abbreviation meaning,0.584,Get an abbreviation meaning,"{""fare basis code"": [""qw""]}","{""fare basis code"": 0.873}",qw,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,4,atis,datasets/atis/processed/filtered/min50/abbreviation_min50.json,abbreviation,Get an abbreviation meaning,what does the fare code f mean,False,f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""fare_basis_code"": [""f""]}","{""fare basis code"": [""f""]}",Get an abbreviation meaning,0.935,Get an abbreviation meaning,"{""fare basis code"": [""f""]}","{""fare basis code"": 1.014}",f,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


## Field Operations

Proposito: adaptar/avaliar os 9 intents do dataset Field Operations. Esperado: 9 arquivos de origem e 2.700 linhas no CSV agregado.

- Fase A (`RUN_BATCH_INFERENCE=False`): gera `evaluation/field_operations/dataset_csv/field_operations_dataset.csv` e CSVs por intent.
- Fase B (`RUN_BATCH_INFERENCE=True`): gera `evaluation/field_operations/results_csv/field_operations_predictions.csv` e CSVs por intent.


In [8]:
field_operations_batch = run_batch_eval_dataset('field_operations', FIELD_OPERATIONS_DATASET_PATHS)


Batch eval dataset: field_operations
Arquivos selecionados: 9
Datasets detectados: ['field_operations']
Dataset view shape: (2700, 25)
Semantic view shape: (2700, 46)
CSV(s) dataset_csv exportados:
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\field_operations_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\access_external_cameras_300_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\confirm_movement_milestone_300_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\control_live_video_stream_300_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\control_location_sharing_300_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\incident_status_300_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\lookup_operational_record_300_dataset.csv
- E:\Intent_Slot_Code\evaluation\field_operations\dataset_csv\medical_evacuation_300_dataset.csv
- 

,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,camera_identifier,camera_search_area,movement_event,location_reference,stream_control_action,location_sharing_action,status,location,target_type,subject,evacuation_mode,casualty_count,severity,resource_type,guidance_artifact_type,procedure_topic,gold_slots_dataset_json,gold_slots_semantic_json
0,0,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Show me the feed fromCAM-2049coveringparking lot,False,CAM-2049,parking lot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""CAM-2049""], ""camera_search_area"": [""parking lot""]}","{""identifier"": [""CAM-2049""], ""search area"": [""parking lot""]}"
1,1,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Can you displayDRN-1107focused onmain entrance,False,DRN-1107,main entrance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""DRN-1107""], ""camera_search_area"": [""main entrance""]}","{""identifier"": [""DRN-1107""], ""search area"": [""main entrance""]}"
2,2,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-5012monitoringside gate,False,BC-5012,side gate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""BC-5012""], ""camera_search_area"": [""side gate""]}","{""identifier"": [""BC-5012""], ""search area"": [""side gate""]}"
3,3,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Access the feed fromDRN-6005watchingmain road,False,DRN-6005,main road,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""DRN-6005""], ""camera_search_area"": [""main road""]}","{""identifier"": [""DRN-6005""], ""search area"": [""main road""]}"
4,4,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-4044coveringwalkway,False,BC-4044,walkway,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""BC-4044""], ""camera_search_area"": [""walkway""]}","{""identifier"": [""BC-4044""], ""search area"": [""walkway""]}"


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,identifier,search area,movement event,location reference,stream control action,location sharing action,status,location,target type,subject,evacuation mode,casualty count,severity,resource type,guidance artifact type,procedure topic,pred__identifier,pred__search area,pred__movement event,pred__location reference,pred__stream control action,pred__location sharing action,pred__status,pred__location,pred__target type,pred__subject,pred__evacuation mode,pred__casualty count,pred__severity,pred__resource type,pred__guidance artifact type,pred__procedure topic
0,0,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Show me the feed fromCAM-2049coveringparking lot,False,"{""camera_identifier"": [""CAM-2049""], ""camera_search_area"": [""parking lot""]}","{""identifier"": [""CAM-2049""], ""search area"": [""parking lot""]}",,0.0,Access External Cameras,{},{},CAM-2049,parking lot,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Can you displayDRN-1107focused onmain entrance,False,"{""camera_identifier"": [""DRN-1107""], ""camera_search_area"": [""main entrance""]}","{""identifier"": [""DRN-1107""], ""search area"": [""main entrance""]}",,0.0,Access External Cameras,{},{},DRN-1107,main entrance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,2,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-5012monitoringside gate,False,"{""camera_identifier"": [""BC-5012""], ""camera_search_area"": [""side gate""]}","{""identifier"": [""BC-5012""], ""search area"": [""side gate""]}",,0.0,Access External Cameras,{},{},BC-5012,side gate,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,3,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Access the feed fromDRN-6005watchingmain road,False,"{""camera_identifier"": [""DRN-6005""], ""camera_search_area"": [""main road""]}","{""identifier"": [""DRN-6005""], ""search area"": [""main road""]}",,0.0,Access External Cameras,{},{},DRN-6005,main road,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,4,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-4044coveringwalkway,False,"{""camera_identifier"": [""BC-4044""], ""camera_search_area"": [""walkway""]}","{""identifier"": [""BC-4044""], ""search area"": [""walkway""]}",,0.0,Access External Cameras,{},{},BC-4044,walkway,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
e:\Intent_Slot_Code\.venv\Lib\site-packages\transformers\pipelines\question_answering.py:390: 

Batch stats: {
  "intent_seconds": 72.01435710000078,
  "slot_seconds": 565.5310054000001,
  "total_seconds": 637.5453625000009,
  "n_rows": 2700,
  "rows_per_second": 4.2349927688478735,
  "group_sizes": {
    "Access External Cameras": 300,
    "Confirm Movement Milestone": 300,
    "Control Live Video Stream": 300,
    "Location Sharing": 300,
    "Incident Status": 300,
    "Lookup Operational Record": 300,
    "Medical Evacuation": 300,
    "Request Additional Support": 300,
    "Request Procedure Guidance": 300
  },
  "tau_intent": 0.25,
  "tau_slots": 0.35,
  "intent_candidate_counts_by_dataset": {
    "field_operations": 9
  }
}
Metrics: {
  "rows": 2700,
  "slot_exact_match": 0.53,
  "joint_intent_slot_exact_match": 0.4674074074074074,
  "slot_exact_match_count": 1431,
  "joint_exact_match_count": 1262
}
CSV(s) results_csv exportados:
- E:\Intent_Slot_Code\evaluation\field_operations\results_csv\field_operations_predictions.csv
- E:\Intent_Slot_Code\evaluation\field_operations

,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,identifier,search area,movement event,location reference,stream control action,location sharing action,status,location,target type,subject,evacuation mode,casualty count,severity,resource type,guidance artifact type,procedure topic,pred__identifier,pred__search area,pred__movement event,pred__location reference,pred__stream control action,pred__location sharing action,pred__status,pred__location,pred__target type,pred__subject,pred__evacuation mode,pred__casualty count,pred__severity,pred__resource type,pred__guidance artifact type,pred__procedure topic
0,0,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Show me the feed fromCAM-2049coveringparking lot,False,"{""camera_identifier"": [""CAM-2049""], ""camera_search_area"": [""parking lot""]}","{""identifier"": [""CAM-2049""], ""search area"": [""parking lot""]}",Access External Cameras,0.501,Access External Cameras,"{""identifier"": [""fromCAM-2049coveringparking""], ""search area"": [""lot""]}","{""identifier"": 1.736, ""search area"": 1.707}",CAM-2049,parking lot,,,,,,,,,,,,,,,fromCAM-2049coveringparking,lot,,,,,,,,,,,,,,
1,1,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Can you displayDRN-1107focused onmain entrance,False,"{""camera_identifier"": [""DRN-1107""], ""camera_search_area"": [""main entrance""]}","{""identifier"": [""DRN-1107""], ""search area"": [""main entrance""]}",Request Additional Support,0.459,Access External Cameras,"{""identifier"": [""displayDRN-1107focused""], ""search area"": [""onmain entrance""]}","{""identifier"": 0.557, ""search area"": 0.677}",DRN-1107,main entrance,,,,,,,,,,,,,,,displayDRN-1107focused,onmain entrance,,,,,,,,,,,,,,
2,2,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-5012monitoringside gate,False,"{""camera_identifier"": [""BC-5012""], ""camera_search_area"": [""side gate""]}","{""identifier"": [""BC-5012""], ""search area"": [""side gate""]}",Control Live Video Stream,0.593,Access External Cameras,"{""identifier"": [""fromBC-5012monitoringside""], ""search area"": [""live stream""]}","{""identifier"": 1.023, ""search area"": 0.933}",BC-5012,side gate,,,,,,,,,,,,,,,fromBC-5012monitoringside,live stream,,,,,,,,,,,,,,
3,3,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Access the feed fromDRN-6005watchingmain road,False,"{""camera_identifier"": [""DRN-6005""], ""camera_search_area"": [""main road""]}","{""identifier"": [""DRN-6005""], ""search area"": [""main road""]}",Access External Cameras,0.525,Access External Cameras,"{""identifier"": [""fromDRN-6005watchingmain""], ""search area"": [""road""]}","{""identifier"": 1.532, ""search area"": 1.254}",DRN-6005,main road,,,,,,,,,,,,,,,fromDRN-6005watchingmain,road,,,,,,,,,,,,,,
4,4,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-4044coveringwalkway,False,"{""camera_identifier"": [""BC-4044""], ""camera_search_area"": [""walkway""]}","{""identifier"": [""BC-4044""], ""search area"": [""walkway""]}",Control Live Video Stream,0.675,Access External Cameras,"{""identifier"": [""fromBC-4044coveringwalkway""], ""search area"": [""live stream""]}","{""identifier"": 1.166, ""search area"": 0.959}",BC-4044,walkway,,,,,,,,,,,,,,,fromBC-4044coveringwalkway,live stream,,,,,,,,,,,,,,


,example_id,dataset_name,source_file,dataset_intent,semantic_intent,sentence,has_repeated_slot_value,camera_identifier,camera_search_area,movement_event,location_reference,stream_control_action,location_sharing_action,status,location,target_type,subject,evacuation_mode,casualty_count,severity,resource_type,guidance_artifact_type,procedure_topic,gold_slots_dataset_json,gold_slots_semantic_json,pred_intent,pred_intent_score,slot_eval_intent,pred_slots_semantic_json,pred_slots_score_json,pred__identifier,pred__search area,pred__movement event,pred__location reference,pred__stream control action,pred__location sharing action,pred__status,pred__location,pred__target type,pred__subject,pred__evacuation mode,pred__casualty count,pred__severity,pred__resource type,pred__guidance artifact type,pred__procedure topic
0,0,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Show me the feed fromCAM-2049coveringparking lot,False,CAM-2049,parking lot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""CAM-2049""], ""camera_search_area"": [""parking lot""]}","{""identifier"": [""CAM-2049""], ""search area"": [""parking lot""]}",Access External Cameras,0.501,Access External Cameras,"{""identifier"": [""fromCAM-2049coveringparking""], ""search area"": [""lot""]}","{""identifier"": 1.736, ""search area"": 1.707}",fromCAM-2049coveringparking,lot,,,,,,,,,,,,,,
1,1,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Can you displayDRN-1107focused onmain entrance,False,DRN-1107,main entrance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""DRN-1107""], ""camera_search_area"": [""main entrance""]}","{""identifier"": [""DRN-1107""], ""search area"": [""main entrance""]}",Request Additional Support,0.459,Access External Cameras,"{""identifier"": [""displayDRN-1107focused""], ""search area"": [""onmain entrance""]}","{""identifier"": 0.557, ""search area"": 0.677}",displayDRN-1107focused,onmain entrance,,,,,,,,,,,,,,
2,2,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-5012monitoringside gate,False,BC-5012,side gate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""BC-5012""], ""camera_search_area"": [""side gate""]}","{""identifier"": [""BC-5012""], ""search area"": [""side gate""]}",Control Live Video Stream,0.593,Access External Cameras,"{""identifier"": [""fromBC-5012monitoringside""], ""search area"": [""live stream""]}","{""identifier"": 1.023, ""search area"": 0.933}",fromBC-5012monitoringside,live stream,,,,,,,,,,,,,,
3,3,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,Access the feed fromDRN-6005watchingmain road,False,DRN-6005,main road,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""DRN-6005""], ""camera_search_area"": [""main road""]}","{""identifier"": [""DRN-6005""], ""search area"": [""main road""]}",Access External Cameras,0.525,Access External Cameras,"{""identifier"": [""fromDRN-6005watchingmain""], ""search area"": [""road""]}","{""identifier"": 1.532, ""search area"": 1.254}",fromDRN-6005watchingmain,road,,,,,,,,,,,,,,
4,4,field_operations,datasets/field_operations/raw/access_external_cameras_300.json,Access External Cameras,Access External Cameras,I need the live stream fromBC-4044coveringwalkway,False,BC-4044,walkway,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""camera_identifier"": [""BC-4044""], ""camera_search_area"": [""walkway""]}","{""identifier"": [""BC-4044""], ""search area"": [""walkway""]}",Control Live Video Stream,0.675,Access External Cameras,"{""identifier"": [""fromBC-4044coveringwalkway""], ""search area"": [""live stream""]}","{""i

In [8]:
# Celula opcional - benchmark legado/oracle.
# Nao faz parte da avaliacao final. So rode se RUN_LEGACY_BASELINE=True e se quiser comparar contra o loop antigo.

def run_legacy_oracle_loop(batch_result, sample_frame: pd.DataFrame):
    rows = []
    start = time.perf_counter()
    processor = get_nlu_processor()
    intent_slot_dict = batch_result['semantic_intent_slot_dict']
    for _, row in sample_frame.iterrows():
        result = intent_slot_pipeline(
            processor,
            row['sentence'],
            intent_slot_dict,
            force_intent=row['semantic_intent'],
        )
        slot_values = {
            slot: [token]
            for slot, (token, _score) in result['slots'].items()
            if token != 'NULL'
        }
        rows.append({
            'pred_intent': result['intent'][0],
            'pred_intent_score': result['intent'][1],
            'pred_slots_semantic_json': json.dumps(slot_values, ensure_ascii=False),
        })
    elapsed = time.perf_counter() - start
    return elapsed, pd.DataFrame(rows)


def benchmark_legacy_oracle(batch_result, sizes=BENCHMARK_SIZES):
    sample_source = batch_result['semantic_view']
    processor = get_nlu_processor()
    for size in sizes:
        n_rows = min(size, len(sample_source))
        if n_rows == 0:
            continue

        sample_frame = sample_source.sample(n=n_rows, random_state=42).reset_index(drop=True)

        batch_start = time.perf_counter()
        _batch_frame, batch_stats = run_semantic_batch_inference(
            processor,
            sample_frame,
            batch_result['semantic_intent_slot_dict'],
            intent_candidates_by_dataset=batch_result['intent_candidates_by_dataset'],
            use_gold_intent_for_slots=True,
            intent_batch_size=INTENT_BATCH_SIZE,
            spacy_batch_size=SPACY_BATCH_SIZE,
            spacy_n_process=SPACY_N_PROCESS,
            qa_batch_size=QA_BATCH_SIZE,
            slot_sentence_batch_size=SLOT_SENTENCE_BATCH_SIZE,
            nli_batch_size=NLI_BATCH_SIZE,
        )
        batch_elapsed = time.perf_counter() - batch_start

        legacy_elapsed, _legacy_frame = run_legacy_oracle_loop(batch_result, sample_frame)
        speedup = (legacy_elapsed / batch_elapsed) if batch_elapsed else float('inf')

        print(
            f'n={n_rows} | batch={batch_elapsed:.2f}s | legacy={legacy_elapsed:.2f}s | speedup={speedup:.2f}x'
        )
        print('batch_stats:', json.dumps(batch_stats, ensure_ascii=False))


if RUN_LEGACY_BASELINE:
    print('Use benchmark_legacy_oracle(snips_batch), benchmark_legacy_oracle(multiwoz_batch), etc. apos rodar a celula do dataset.')
else:
    print('Benchmark legado desativado. Defina RUN_LEGACY_BASELINE=True e chame benchmark_legacy_oracle(<resultado>).')


Benchmark legado desativado. Defina RUN_LEGACY_BASELINE=True e chame benchmark_legacy_oracle(<resultado>).
